# Customer Retention Intelligence — Data Ingestion

## Project Objective
Predict which customers are likely to churn within the next 30 days, so the
business can identify high-risk customers early and trigger the right
retention action before they leave.

## Data Sources
This notebook ingests and validates five raw synthetic datasets:

- `customers.csv` — customer profile and signup information
- `transactions.csv` — billing / payment history
- `usage.csv` — longitudinal product engagement (logins, sessions, feature usage)
- `support_tickets.csv` — customer support interactions
- `churn_events.csv` — churn ground-truth events

## Prediction Target (planned for a later phase)
`churn_next_30_days` will be a binary label built later, during feature
engineering, from `churn_events.csv` relative to a chosen prediction date:

```
churn_next_30_days = 1  if  prediction_date < churn_date <= prediction_date + 30 days
churn_next_30_days = 0  otherwise
```

It is **not** created in this notebook — this notebook only ingests and
validates the raw event tables that later feed that calculation.

## Why Validation Matters
Before any feature engineering or modeling happens, we need confidence that:

- every dataset has the expected schema and column types,
- foreign keys (`customer_id`) are consistent across every table,
- event dates are chronologically sensible relative to each customer's `signup_date`,
- "missing" values are the *intentional* synthetic missingness designed into
  the generator, not silent data corruption.

Skipping this step risks building features and models on broken data and
only discovering it much later, when it is far more expensive to fix.

## Synthetic-Data Limitation
All five datasets are synthetically generated (see
`src/data/generate_data.py`). They are designed with realistic, noisy
relationships (e.g. engagement vs. churn risk, ticket priority vs.
resolution time), but they do not capture the full complexity of real
customer behavior. Conclusions drawn later in this project should be read
as a methodology demonstration, not as real business insight.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Notebooks run with their working directory set to notebooks/, so resolve
# the project root explicitly and make src/ importable regardless of how
# this notebook is launched.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 20)
PROJECT_ROOT


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## Load Configuration

In [2]:
import yaml

with open(PROJECT_ROOT / "configs" / "data_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)

config


{'random_seed': 42,
 'num_customers': 10000,
 'start_date': '2024-01-01',
 'end_date': '2026-06-30',
 'missing_value_rate': 0.03,
 'output_directory': 'data/raw'}

## Load Raw Datasets

In [3]:
from src.data.load_data import load_all_datasets

datasets = load_all_datasets()

for name, df in datasets.items():
    print(f"{name:16s} shape={df.shape}")


customers        shape=(10000, 7)
transactions     shape=(299921, 6)
usage            shape=(1246888, 7)
support_tickets  shape=(85577, 8)
churn_events     shape=(1929, 3)


In [4]:
for name, df in datasets.items():
    print(f"\n=== {name} ===")
    print("Columns:", list(df.columns))
    print(df.dtypes)



=== customers ===
Columns: ['customer_id', 'signup_date', 'age', 'country', 'plan', 'contract_type', 'acquisition_channel']
customer_id                    object
signup_date            datetime64[ns]
age                             int64
country                        object
plan                           object
contract_type                  object
acquisition_channel            object
dtype: object

=== transactions ===
Columns: ['customer_id', 'transaction_date', 'transaction_type', 'amount', 'payment_method', 'payment_status']
customer_id                 object
transaction_date    datetime64[ns]
transaction_type            object
amount                     float64
payment_method              object
payment_status              object
dtype: object

=== usage ===
Columns: ['customer_id', 'usage_date', 'login_count', 'session_minutes', 'feature_usage_count', 'pages_viewed', 'active_minutes']
customer_id                    object
usage_date             datetime64[ns]
login_count      

## Basic Data Profiling

For each dataset: shape, a small sample of rows, `info()`, missing-value
counts/percentages, and a compact `describe()` summary. `describe()` and
`info()` produce small, fixed-size summaries regardless of row count, so
this stays readable even for the 1.2M-row usage table.


In [5]:
def profile_dataset(name: str, df: pd.DataFrame) -> None:
    """Print a compact profile of one dataset: shape, sample, info, missingness, describe."""
    print(f"\n{'=' * 70}\n{name.upper()}  shape={df.shape}\n{'=' * 70}")

    display(df.head(3))

    df.info()

    missing_counts = df.isna().sum()
    missing_pct = (missing_counts / len(df) * 100).round(2)
    missing_summary = pd.DataFrame(
        {"missing_count": missing_counts, "missing_pct": missing_pct}
    )
    missing_summary = missing_summary[missing_summary["missing_count"] > 0]

    if missing_summary.empty:
        print("No missing values.")
    else:
        print("Missing values:")
        display(missing_summary)

    display(df.describe(include="all").T)


for name, df in datasets.items():
    profile_dataset(name, df)



CUSTOMERS  shape=(10000, 7)


,customer_id,signup_date,age,country,plan,contract_type,acquisition_channel
0,C000001,2025-05-10,69,United States,Basic,Monthly,Referral
1,C000002,2025-12-31,24,United Kingdom,Standard,Monthly,Social Media
2,C000003,2025-11-10,25,United States,Standard,Monthly,Organic


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          10000 non-null  object        
 1   signup_date          10000 non-null  datetime64[ns]
 2   age                  10000 non-null  int64         
 3   country              10000 non-null  object        
 4   plan                 10000 non-null  object        
 5   contract_type        10000 non-null  object        
 6   acquisition_channel  10000 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 547.0+ KB
No missing values.


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,10000,10000,C000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
signup_date,10000,NaN,NaN,NaN,2024-07-08 03:31:58.080000,2023-01-01 00:00:00,2023-10-09 00:00:00,2024-07-12 00:00:00,2025-04-09 00:00:00,2025-12-31 00:00:00,NaN
age,10000.0,NaN,NaN,NaN,43.4459,18.0,31.0,43.0,57.0,69.0,15.102599
country,10000,5,India,3987,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plan,10000,3,Basic,5041,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contract_type,10000,2,Monthly,7020,NaN,NaN,NaN,NaN,NaN,NaN,NaN
acquisition_channel,10000,5,Organic,3019,NaN,NaN,NaN,NaN,NaN,NaN,NaN



TRANSACTIONS  shape=(299921, 6)


,customer_id,transaction_date,transaction_type,amount,payment_method,payment_status
0,C000001,2025-12-23,Purchase,20.02,UPI,Success
1,C000001,2025-12-28,Purchase,27.23,Credit Card,Success
2,C000001,2025-06-28,Renewal,28.20,UPI,Success


... (13 outputs left out)


CHURN_EVENTS  shape=(1929, 3)


,customer_id,churn_date,churn_reason
0,C000019,2025-12-13,Customer Choice
1,C000020,2025-09-17,Customer Choice
2,C000042,2026-05-27,Customer Choice


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1929 entries, 0 to 1928
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   customer_id   1929 non-null   object        
 1   churn_date    1929 non-null   datetime64[ns]
 2   churn_reason  1929 non-null   object        
dtypes: datetime64[ns](1), object(2)
memory usage: 45.3+ KB
No missing values.


,count,unique,top,freq,mean,min,25%,50%,75%,max
customer_id,1929,1929,C000019,1,NaN,NaN,NaN,NaN,NaN,NaN
churn_date,1929,NaN,NaN,NaN,2025-10-09 17:52:43.297044992,2023-06-05 00:00:00,2025-06-26 00:00:00,2025-12-03 00:00:00,2026-03-16 00:00:00,2026-06-27 00:00:00
churn_reason,1929,6,Customer Choice,496,NaN,NaN,NaN,NaN,NaN,NaN


## Data Validation

Runs the full schema / datatype / data-quality checks from
`src/data/validate_data.py` for every dataset. ~3% missing values in the
intended usage and support-ticket metric columns are expected and are
*not* treated as validation failures.


In [6]:
from src.data.validate_data import validate_all

validation_output = validate_all(
    datasets["customers"],
    datasets["transactions"],
    datasets["usage"],
    datasets["support_tickets"],
    datasets["churn_events"],
)

validation_output["report"]


,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Invalid IDs,Invalid Dates,Invalid Values,Errors,Warnings,Status
0,customers,10000,7,0,0,0,0,0,0,0,PASS
1,transactions,299921,6,0,0,0,0,0,0,0,PASS
2,usage,1246888,7,188284,0,0,0,0,0,0,PASS
3,support_tickets,85577,8,5231,0,0,0,0,0,0,PASS
4,churn_events,1929,3,0,0,0,0,0,0,0,PASS


In [7]:
for result in validation_output["results"]:
    if result.errors or result.warnings:
        print(f"\n[{result.dataset}]")
        for message in result.errors:
            print("  ERROR:", message)
        for message in result.warnings:
            print("  WARNING:", message)
    else:
        print(f"[{result.dataset}] no errors or warnings")


[customers] no errors or warnings
[transactions] no errors or warnings
[usage] no errors or warnings
[support_tickets] no errors or warnings
[churn_events] no errors or warnings


## Relationship Validation

Checks that every `customer_id` referenced in `transactions`, `usage`,
`support_tickets`, and `churn_events` exists in `customers.csv`. Expected
result: 0 orphan records everywhere.


In [8]:
pd.DataFrame([validation_output["relationships"]])


,orphan_transactions,orphan_usage_records,orphan_support_tickets,orphan_churn_events,total_orphans
0,0,0,0,0,0


## Date Validation

Cross-table checks that every event date respects its customer's
`signup_date` and that no date falls after the project end date
(`2026-06-30`).


In [9]:
pd.DataFrame([validation_output["date_consistency"]])


,transaction_date_before_signup,usage_date_before_signup,created_date_before_signup,churn_date_not_after_signup,transaction_date_after_max,usage_date_after_max,created_date_after_max,churn_date_after_max
0,0,0,0,0,0,0,0,0


## Business Rule Validation

Business-level statistics: negative values that shouldn't exist, the
realized churn rate, and how many customers have zero activity in each
table. These are informational, not pass/fail checks.


In [10]:
pd.DataFrame([validation_output["business_rules"]])


,total_customers,total_transactions,negative_transaction_amounts,failed_payment_rate_pct,negative_usage_metric_values,satisfaction_score_out_of_bounds,negative_resolution_time_hours,customers_with_multiple_churn_events,churn_rate_pct,customers_with_zero_transactions,customers_with_zero_usage_records,customers_with_zero_support_tickets,customers_with_churn_event
0,10000,299921,0,4.98,0,0,0,0,19.29,0,0,2411,1929


## Save Processed Data

Saves validated datasets to `data/processed/` as Parquet files. The raw
CSV files in `data/raw/` are never modified. This step only runs if
validation passed.


In [11]:
from src.data.load_data import save_processed_datasets

assert validation_output["passed"], "Validation must pass before saving processed data."

saved_paths = save_processed_datasets(datasets)
saved_paths


{'customers': PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/customers.parquet'),
 'transactions': PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/transactions.parquet'),
 'usage': PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/usage.parquet'),
 'support_tickets': PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/support_tickets.parquet'),
 'churn_events': PosixPath('/Users/dhananjaymandal/customer-retention-intelligence/data/processed/churn_events.parquet')}

## Final Validation Summary

In [12]:
summary = validation_output["report"][["Dataset", "Rows", "Columns", "Missing Values", "Status"]]
summary = summary.rename(columns={"Missing Values": "Missing", "Status": "Validation Status"})
summary


,Dataset,Rows,Columns,Missing,Validation Status
0,customers,10000,7,0,PASS
1,transactions,299921,6,0,PASS
2,usage,1246888,7,188284,PASS
3,support_tickets,85577,8,5231,PASS
4,churn_events,1929,3,0,PASS
